# Control clásico frente a aprendizaje por refuerzo en el péndulo invertido (CartPole-v1)

**Versión completa.** Sigue el orden de la charla: **entorno → control clásico → aprendizaje por
refuerzo → comparación**. Recalcula en vivo lo que cabe en una sesión:

| Qué | En vivo | Bandera |
|---|---|---|
| Linealización, LQR, MPC y todas las simulaciones | siempre (segundos) | — |
| Sintonización del PID y del fuzzy (evolución diferencial, 3 ejecuciones cada uno) | sí | `RESINTONIZAR` |
| Q-learning, SARSA, DQN y actor-crítico: **una** semilla cada uno, con la configuración del registro | sí | `REENTRENAR` |
| Evaluación comparativa completa (10 semillas por método de RL, 1728 tareas) | no: se carga de `results/` | `REEVALUAR` |

La celda de configuración imprime cuánto tardó cada paso en la máquina en que se generaron los
registros. La versión corta, `02_charla_resumen.ipynb`, no entrena ni sintoniza nada: solo carga los
resultados.

**Reproducibilidad.** El registro se generó en Windows (`results/rl/*.json`, campo `versions`). En esa
máquina, los scripts con `--regenerate-artifacts` comprueban que una semilla se reproduce exactamente.
En otro sistema operativo, el redondeo de las funciones matemáticas puede ser distinto, y un
entrenamiento de RL es caótico: la misma semilla puede seguir otra trayectoria. En una prueba en Linux,
Q-learning, DQN y actor-crítico **no** reprodujeron bit a bit el registro (no se ha aislado la causa).
Por eso el notebook no exige igualdad: compara la ejecución en vivo con las 10 semillas del registro e
informa de si coincide.

Las justificaciones de cada decisión están en `docs/`. El texto las resume y remite a ellas. Los números
los imprime el código. Los que se citan del registro se marcan como tales.

In [ ]:
# Importaciones y estilo comunes. Todo lo que no es trivial vive en el paquete cartpole_lab
# (probado en tests/), para que las celdas se lean durante la charla.
import json
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from cartpole_lab import CartPoleTask, ForcePulse, continuous_dynamics, load_params, run_episode
from cartpole_lab import teaching as tl
from cartpole_lab.linearization import continuous_linearization, controllability_matrix, euler_discretization
from cartpole_lab.plotting import METHOD_COLORS, MUTED, SERIES, apply_style
from cartpole_lab.sanity import NEAR_BOUNDARY_INITIAL_STATES, REFERENCE_HARD_INITIAL_STATE, sanity_statistics

params = load_params()
apply_style()

In [ ]:
# Qué se recalcula (ver la tabla de arriba). Con las tres primeras en True, la duración la marcan los
# entrenamientos y sintonizaciones (tiempos de la máquina del registro, impresos abajo).
RESINTONIZAR = True   # PID y fuzzy
REENTRENAR = True     # una semilla de cada método de RL
REEVALUAR = False     # relanza scripts/evaluate_all.py y scripts/make_figures.py (multiproceso)

tab = tl.load_result("rl/tabular.json")
tiempos = {
    "Sintonización del PID (3 ejecuciones)": tl.load_result("tuning/pid.json")["tuning"]["wall_time_s"],
    "Sintonización del fuzzy (3 ejecuciones)": tl.load_result("tuning/fuzzy.json")["tuning"]["wall_time_s"],
    "Q-learning, 1 semilla": np.mean([r["wall_time_s"] for r in tab["runs"]
                                      if r["method"] == "q_learning" and r["discretization"] == "boxes_1983"]),
    "SARSA, 1 semilla": np.mean([r["wall_time_s"] for r in tab["runs"]
                                 if r["method"] == "sarsa" and r["discretization"] == "boxes_1983"]),
    "DQN, 1 semilla": tl.load_result("rl/dqn.json")["summary"]["dqn"]["wall_time_per_run_s"]["mean"],
    "Actor-crítico, 1 semilla": tl.load_result("rl/actor_critic.json")["summary"]["wall_time_per_run_s"]["mean"],
}
print("Tiempos REGISTRADOS (máquina del registro, no medidos aquí):")
for k, v in tiempos.items():
    print(f"  {k:<42} {v:6.0f} s")
print(f"  {'Total con RESINTONIZAR y REENTRENAR':<42} {sum(tiempos.values()) / 60:6.1f} min")

## 1. El entorno: CartPole-v1 tal como lo simula Gymnasium (doc 01)

Un carro sobre un raíl sin rozamiento y un poste articulado que hay que mantener vertical. Estado
$s = [x, \dot x, \theta, \dot\theta]$. El episodio termina si $|\theta| > 12°$ o $|x| > 2.4$ m, y
se trunca a los 500 pasos (10 s). **Todos los métodos usan la misma física, el mismo estado, la misma
terminación y el mismo bucle de simulación** (`CartPoleTask` y `run_episode`). La única diferencia
permitida es el actuador: los clásicos piden una fuerza continua en $[-10, 10]$ N y el RL tabular y el
DQN eligen entre 5 niveles $\{-10, -5, 0, 5, 10\}$ N (docs/01 §7.1).

In [ ]:
# Constantes físicas LEÍDAS del simulador (nunca copiadas a mano; docs/01 §2).
for name, value in vars(params).items():
    print(f"{name:<24} {value}")
print(f"\nθ límite = {np.degrees(params.theta_threshold_radians):.1f}°   ·   "
      f"τ = {params.tau} s ({1 / params.tau:.0f} Hz)   ·   episodio = 500 pasos = 10 s")

In [ ]:
# Convenciones de signo que dan por supuestas LQR y MPC (docs/01 §3), comprobadas sobre el modelo.
empuje = continuous_dynamics(np.zeros(4), +params.force_mag, params)
inclinado = continuous_dynamics([0.0, 0.0, 0.05, 0.0], 0.0, params)
print("F > 0 acelera el carro hacia +x (ẍ > 0):            ", bool(empuje[1] > 0))
print("F > 0 hace girar el poste hacia θ < 0 (θ̈ < 0):     ", bool(empuje[3] < 0))
print("Con F = 0, un θ > 0 crece (equilibrio inestable):   ", bool(inclinado[3] > 0))
display(Image(filename=str(tl.figure_path("00_convencion_signos.png")), width=720))

Cerca de la vertical, el sistema se aproxima por $\dot s = A_c s + B_c F$. El carro es un doble
integrador; el poste tiene un modo inestable en $+\sqrt a$. El simulador integra con **Euler
explícito** ($\tau = 0.02$ s), así que el modelo discreto exacto del simulador es
$A_d = I + \tau A_c$, $B_d = \tau B_c$ (docs/01 §5 y docs/04 §3).

In [ ]:
# Linealización en el equilibrio vertical (docs/04 §2): de aquí salen el modo inestable y el modelo de LQR/MPC.
A_c, B_c = continuous_linearization(params)
A_d, B_d = euler_discretization(A_c, B_c, params.tau)
polo = float(np.max(np.linalg.eigvals(A_c).real))
print("autovalores de A_c:", np.round(np.sort(np.linalg.eigvals(A_c).real), 3))
print(f"modo inestable +√a = {polo:.2f} 1/s  →  sin control, un error se multiplica por e cada {1 / polo:.2f} s")
print("rango de la matriz de controlabilidad (Euler):", np.linalg.matrix_rank(controllability_matrix(A_d, B_d)), "de 4")

In [ ]:
# Sin control (F = 0) desde θ0 = 0.01 rad: ¿cuánto tarda en caer, y cuánto acierta el modelo lineal?
class FuerzaNula:
    """Controlador trivial (F = 0): fija la escala de tiempo del problema."""
    action_mode = "continuous"
    def reset(self): pass
    def __call__(self, state): return 0.0

s0 = np.array([0.0, 0.0, 0.01, 0.0])
caida = tl.simulate({"Sin control": FuerzaNula()}, initial_state=s0)["Sin control"]
lineal = np.array([np.linalg.matrix_power(A_d, k) @ s0 for k in range(caida.n_steps + 1)])
print(f"{caida.termination_reason} tras {caida.n_steps} pasos = {caida.duration:.2f} s")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(caida.times, np.degrees(caida.states[:, 2]), color=SERIES[0], label="simulador (no lineal)")
ax.plot(caida.times, np.degrees(lineal[:, 2]), "--", color=SERIES[1], label="modelo lineal de Euler (LQR/MPC)")
ax.axhline(np.degrees(params.theta_threshold_radians), color=MUTED, linestyle=":")
ax.set_xlabel("t [s]"); ax.set_ylabel("θ [°]"); ax.legend(loc="upper left")
ax.set_title("Sin control el poste cae en menos de un segundo", loc="left", fontsize=11)
plt.show()

## 2. Control clásico

Cuatro controladores, de menos a más uso del modelo: **PID** (caja negra, sintonizado sobre el
simulador), **LQR** (modelo lineal + criterio cuadrático), **MPC** (modelo lineal + restricciones) y
**fuzzy** (reglas escritas por un experto). Todos se comprueban con el mismo *sanity check*: 50 CI de la
distribución oficial (semillas 1000–1049) y el estado difícil de referencia
$s_0 = (0.5, 0.2, 0.1, 0.2)$ (`sanity.py`).

### 2.1 PID en cascada (doc 03)

Un PID sobre $\theta$ mantiene el poste, pero deja el carro a la deriva. Por eso se usa una **cascada**:
un lazo externo PD sobre $x$ pide una inclinación $\theta_{ref}$ (como mucho ±3°) y un lazo interno PID
la sigue. **Ziegler–Nichols no es aplicable**: con control P puro no existe una ganancia "última" con
oscilación sostenida, porque el lazo discreto es inestable para **toda** $K_p$.

In [ ]:
# Ziegler–Nichols exige una Kp con oscilación sostenida (radio espectral = 1 exacto en el límite).
# Aquí el radio espectral del lazo P sobre θ es ≥ 1 para toda Kp (docs/03 §3.1).
from cartpole_lab.controllers.pid_tuning import p_only_angle_spectral_radius

kps = np.linspace(0.0, 200.0, 401)
rho = np.array([p_only_angle_spectral_radius(kp, params) for kp in kps])
print(f"radio espectral mínimo con P puro en Kp ∈ [0, 200] N/rad: {rho.min():.4f} (en Kp = {kps[rho.argmin()]:.1f})")
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(kps, rho, color=METHOD_COLORS["PID"])
ax.axhline(1.0, color=MUTED, linestyle=":")
ax.set_xlabel("Kp [N/rad]"); ax.set_ylabel("radio espectral")
ax.set_title("P puro sobre θ: ninguna ganancia estabiliza el lazo (≥ 1 siempre)", loc="left", fontsize=11)
plt.show()

**Sintonización por optimización** (docs/03 §3.3): evolución diferencial de SciPy sobre un criterio
ITAE + esfuerzo, evaluado por simulación en 32 CI, con 3 ejecuciones independientes del optimizador.
Es reproducible, el criterio está escrito y **no usa el modelo**: el modelo queda para el LQR.

In [ ]:
from cartpole_lab.controllers.pid import CascadePID, PIDGains, load_tuning_record
from cartpole_lab.controllers.pid_tuning import TuningConfig, closed_loop_eigenvalues, tune_pid

registro_pid = load_tuning_record()
config_pid = TuningConfig()
if RESINTONIZAR:
    t0 = time.perf_counter()
    resultado = tune_pid(config_pid, params)
    print(f"{len(resultado.runs)} ejecuciones de evolución diferencial en {time.perf_counter() - t0:.0f} s")
    for run in resultado.runs:
        print(f"  semilla {run.de_seed}: J = {run.cost:.5f}  " +
              ", ".join(f"{k} = {v:.4g}" for k, v in run.gains.to_dict().items()))
    ganancias_pid = resultado.best.gains
else:
    ganancias_pid = PIDGains(**registro_pid["gains"])

print("\nGanancias elegidas (en vivo | registro):")
for k, v in ganancias_pid.to_dict().items():
    print(f"  {k:<9} {v:12.6g} | {registro_pid['gains'][k]:12.6g}")
print("Ganancias:", tl.compare_with_record(list(ganancias_pid.to_dict().values()),
                                             list(registro_pid["gains"].values()), rtol=1e-6))

**Hallazgo** (docs/03 §4.2): el optimizador **anula el término integral** ($K_i \approx 0$). Sin
rozamiento ni fuerzas constantes no hay error estacionario que eliminar. La I solo añade un modo lento y
retardo. *En este sistema, el mejor PID es un PD.* Los modos físicos del lazo linealizado lo confirman:

In [ ]:
limite = config_pid.theta_ref_limit(params)
fisicos = np.abs(closed_loop_eigenvalues(ganancias_pid, params, limite, include_integral=False))
print("|λ| de los modos físicos (PD):", np.round(np.sort(fisicos), 4))
print(f"constante de tiempo dominante: {-params.tau / np.log(fisicos.max()):.2f} s")

pid = CascadePID(ganancias_pid, params)
sanity = sanity_statistics(pid, params)
print(f"sanity check (50 CI oficiales): sobrevive {sanity['survival_rate']:.0%}, "
      f"estabilizado {sanity['stabilized_rate']:.0%}, esfuerzo medio {sanity['control_effort_abs_Ns']['mean']:.2f} N·s")
tray = tl.simulate({"PID": pid}, initial_state=REFERENCE_HARD_INITIAL_STATE)
tl.plot_trajectories(tray, params, title="PID en cascada desde el estado difícil de referencia")
plt.show()

### 2.2 LQR (doc 04)

Con el modelo lineal $s_{k+1} = A_d s_k + B_d F_k$, el control que minimiza
$J = \sum_k s_k^\top Q s_k + R F_k^2$ es $F = -K s$, con $K$ dada por la ecuación de Riccati discreta.
**Q y R por la regla de Bryson** (Bryson & Ho, 1975): $Q_{ii} = 1/(\text{máximo aceptable de } s_i)^2$,
$R = 1/F_{max}^2$. Con θ aceptable = 8.1° (validez del modelo lineal, $\cos\theta \ge 0.99$) y
x aceptable = 1.2 m (supuesto: la mitad del semirraíl). Diseñarlo cuesta milisegundos y **ningún
episodio de prueba**.

In [ ]:
from cartpole_lab.controllers.lqr import LQRController, design_default_lqr, lqr_criterion

t0 = time.perf_counter()
diseno = design_default_lqr(params)
print(f"diseño en {1e3 * (time.perf_counter() - t0):.1f} ms")
print("diag(Q) =", np.round(np.diag(diseno.Q), 4), "  R =", diseno.R[0, 0])
print("K =", np.round(diseno.K[0], 3), "   (F = −K s)")
print("K:", tl.compare_with_record(diseno.K, tl.load_result("tuning/lqr.json")["K"]))
print(f"radio espectral en lazo cerrado {diseno.spectral_radius:.4f} → constante de tiempo dominante "
      f"{diseno.dominant_time_constant(params.tau):.2f} s")
g = ganancias_pid
print("Realimentación equivalente del PID (Ki ≈ 0):",
      np.round([g.kp_theta * g.kp_x, g.kp_theta * g.kd_x, g.kp_theta, g.kd_theta], 3),
      "frente a −K del LQR:", np.round(-diseno.K[0], 3))

PID y LQR tienen la **misma forma** (realimentación de estado); solo difiere cómo se eligen las
ganancias. ¿Cuál es "mejor"? Cada uno se evalúa con el criterio del otro sobre 32 CI nuevas
(semilla 2027, fuera de muestra para ambos), en comparación **pareada** (docs/04 §5.3):

In [ ]:
from cartpole_lab.controllers.pid_tuning import itae_cost

lqr = LQRController(diseno.K, params)
caja = np.asarray(config_pid.initial_state_box)
cis = np.random.default_rng(2027).uniform(-caja, caja, size=(config_pid.n_initial_states, 4))
criterios = {}
for nombre, c in {"PID": pid, "LQR": lqr}.items():
    trs = [tl.simulate({nombre: c}, initial_state=s)[nombre] for s in cis]
    criterios[nombre] = (np.array([itae_cost(t, params, config_pid) for t in trs]),
                         np.array([lqr_criterion(t, diseno.Q, diseno.R) for t in trs]))
(pid_itae, pid_quad), (lqr_itae, lqr_quad) = criterios["PID"], criterios["LQR"]
print(f"criterio ITAE (el del PID):        gana el PID en {np.sum(pid_itae < lqr_itae)}/{len(cis)} CI")
print(f"criterio cuadrático (el del LQR):  gana el LQR en {np.sum(lqr_quad < pid_quad)}/{len(cis)} CI")
print("\n→ Cada método gana con su propio criterio: 'mejor' no tiene sentido sin decir 'según qué'.")

tray = tl.simulate({"PID": pid, "LQR": lqr}, initial_state=REFERENCE_HARD_INITIAL_STATE)
tl.plot_trajectories(tray, params, title="LQR frente a PID desde el estado difícil de referencia")
plt.show()

### 2.3 MPC lineal (doc 05)

El LQR no "ve" los límites: lejos del equilibrio satura y puede perder el poste. El MPC resuelve **en
cada paso** (cada 20 ms) un problema de optimización con el mismo modelo, los mismos Q y R y coste
terminal P del LQR, más **restricciones explícitas**: $|F| \le 10$ N (dura) y $|\theta| \le 8.1°$,
$|x| \le 2.4$ m (blandas, penalización L1 exacta). Horizonte N = 25 (0.5 s), elegido con datos.
Solver: CVXPY + Clarabel.

Consecuencia comprobable: **si ninguna restricción está activa, el MPC coincide exactamente con el LQR**.

In [ ]:
from cartpole_lab.controllers.mpc import LinearMPC

mpc = LinearMPC(params)
s_pequeno = np.array([0.1, 0.0, 0.02, 0.0])
print(f"estado pequeño {s_pequeno}:  F_MPC = {mpc(s_pequeno):.6f} N,  F_LQR = {lqr(s_pequeno):.6f} N")

s_dificil = np.array([0.0, 0.0, 0.10, 1.25])  # poste inclinado y cayendo deprisa (estado cerca de la frontera)
mpc(s_dificil)
plan = mpc.last_plan
t_plan = np.arange(plan["states"].shape[0]) * params.tau
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(t_plan, np.degrees(plan["states"][:, 2]), "o-", color=METHOD_COLORS["MPC"], markersize=3,
        label="θ planificado por el MPC")
ax.axhline(np.degrees(mpc.config.theta_constraint(params)), color=MUTED, linestyle="--", label="restricción 8.1°")
ax.axhline(np.degrees(params.theta_threshold_radians), color=MUTED, linestyle=":", label="fallo 12°")
ax.set_xlabel("horizonte de predicción [s]"); ax.set_ylabel("θ [°]"); ax.legend(loc="upper right")
ax.set_title(f"Plan del MPC desde s0 = {s_dificil}: planifica CON los límites", loc="left", fontsize=11)
plt.show()

In [ ]:
# Los 12 estados recuperables cerca de la frontera (sanity.py): ¿quién sobrevive?
# El MPC resuelve un problema por paso: esta celda tarda unos segundos.
tiempos_mpc, sobrevive = [], {"PID": [], "LQR": [], "MPC": []}
t0 = time.perf_counter()
for s0 in NEAR_BOUNDARY_INITIAL_STATES:
    for nombre, c in {"PID": pid, "LQR": lqr, "MPC": mpc}.items():
        tr = tl.simulate({nombre: c}, initial_state=s0)[nombre]
        sobrevive[nombre].append(tr.termination_reason == "time_limit")
    tiempos_mpc += mpc.solve_times
print(f"{3 * len(NEAR_BOUNDARY_INITIAL_STATES)} episodios en {time.perf_counter() - t0:.0f} s")
for nombre, v in sobrevive.items():
    print(f"  {nombre}: sobrevive {sum(v)}/{len(v)}   ({tl.compare_with_record(v, tl.load_result('tuning/mpc.json')['near_boundary_comparison'][nombre])})")
t_ms = 1e3 * np.array(tiempos_mpc)
print(f"\nTiempo por resolución del MPC en ESTA máquina: media {t_ms.mean():.1f} ms, máximo {t_ms.max():.1f} ms "
      f"(periodo de control: 20 ms; en tiempo real importa el peor caso)")

tray = tl.simulate({"PID": pid, "LQR": lqr, "MPC": mpc}, initial_state=s_dificil)
tl.plot_trajectories(tray, params, title=f"Cerca de la frontera, s0 = {s_dificil}")
plt.show()

### 2.4 Fuzzy Takagi-Sugeno de orden 0 (doc 06)

La misma cascada que el PID, pero el lazo interno se escribe con **25 reglas** en lenguaje natural
("si el poste cae a la derecha y deprisa, empuja fuerte a la derecha"). Membresías triangulares y tabla
de MacVicar-Whelan (1981), **fijas**. Solo se sintonizan 3 factores de escala y el lazo externo, con
**exactamente** el protocolo del PID. Consecuencia demostrable: en la celda central el fuzzy **es un
PD**, y fuera de ella satura suavemente.

In [ ]:
from cartpole_lab.controllers.fuzzy import (LABELS, RULE_TABLE, CascadeFuzzy, FuzzyGains, memberships,
                                            load_tuning_record as load_fuzzy_record)
from cartpole_lab.controllers.fuzzy_tuning import FuzzyTuningConfig, tune_fuzzy

print("Tabla de reglas (consecuente en unidades de U; filas: e = θ − θ_ref, columnas: θ̇):")
print("       " + "".join(f"{l:>6}" for l in LABELS))
for etiqueta, fila in zip(LABELS, RULE_TABLE):
    print(f"  {etiqueta:<5}" + "".join(f"{v:>6.1f}" for v in fila))

z = np.linspace(-1.4, 1.4, 400)
fig, ax = plt.subplots(figsize=(8, 2.8))
for k, etiqueta in enumerate(LABELS):
    ax.plot(z, memberships(z)[:, k], color=METHOD_COLORS["Fuzzy"])
    ax.text(-1 + 0.5 * k, 1.05, etiqueta, ha="center")
ax.set_ylim(0, 1.2); ax.set_xlabel("entrada normalizada (e/E o θ̇/D)"); ax.set_ylabel("pertenencia")
ax.set_title("Membresías triangulares: partición de la unidad", loc="left", fontsize=11)
plt.show()

In [ ]:
registro_fuzzy = load_fuzzy_record()
if RESINTONIZAR:
    t0 = time.perf_counter()
    resultado_f = tune_fuzzy(FuzzyTuningConfig(), params)
    print(f"{len(resultado_f.runs)} ejecuciones en {time.perf_counter() - t0:.0f} s; J = "
          + ", ".join(f"{r.cost:.5f}" for r in resultado_f.runs))
    ganancias_fuzzy = resultado_f.best.gains
else:
    ganancias_fuzzy = FuzzyGains(**registro_fuzzy["gains"])
print("Factores de escala y lazo externo:", {k: round(v, 4) for k, v in ganancias_fuzzy.to_dict().items()})
print("Parámetros:", tl.compare_with_record(list(ganancias_fuzzy.to_dict().values()),
                                            list(registro_fuzzy["gains"].values()), rtol=1e-6))
pd = ganancias_fuzzy.equivalent_pd()
print(f"PD equivalente en la celda central: Kp_θ = {pd['kp_theta']:.1f} N/rad, Kd_θ = {pd['kd_theta']:.1f} N·s/rad "
      f"(PID: {ganancias_pid.kp_theta:.1f} y {ganancias_pid.kd_theta:.1f})")

fuzzy = CascadeFuzzy(ganancias_fuzzy, params)
tl.plot_force_maps({"PID (integral a 0)": pid, "Fuzzy": fuzzy, "LQR": lqr}, params,
                   title="Qué fuerza pide cada controlador con el carro centrado y quieto")
tray = tl.simulate({"PID": pid, "Fuzzy": fuzzy}, initial_state=REFERENCE_HARD_INITIAL_STATE)
tl.plot_trajectories(tray, params, title="Fuzzy frente a PID desde el estado difícil de referencia")
plt.show()

**El fuzzy es un lenguaje, no una ley de control nueva** (docs/06 §6): lo que se escribe con reglas es un
PD con saturación suave. Con el mismo presupuesto de sintonización queda ligeramente por detrás del PID
en su criterio (registro: 4.7 % de media en 32 CI fuera de muestra). Su valor está en que la lógica es
legible.

## 3. Aprendizaje por refuerzo

Mismo entorno y misma recompensa de CartPole-v1 (+1 por paso vivo), **sin modelo ni ecuaciones**: el
agente aprende por ensayo y error. Protocolo común (docs/07_08, 09 y 10): 10 semillas por método;
**validación** en las CI 2000–2009 (para elegir la mejor política vista); **test** en las CI 1000–1049
(las mismas del *sanity check* de los clásicos). Umbral de "resuelto": 475, el `reward_threshold`
oficial de CartPole-v1.

Con `REENTRENAR = True` se entrena **una** semilla (la exportada para el navegador) con la configuración
exacta del registro, y se compara con las 10 semillas registradas. Con `False` se carga su política
guardada.

### 3.1 Q-learning y SARSA tabulares (doc 07_08)

El estado continuo se agrupa en **162 cajas**, la partición de Barto, Sutton y Anderson (1983),
verificada en el código original `pole.c`: fina cerca de la vertical, gruesa en el resto. Se aprende un
valor $Q(s, a)$ por caja y acción con diferencias temporales. Q-learning y SARSA solo difieren en el
objetivo: $\max_{a'} Q(s', a')$ (*off-policy*) frente a la acción que de verdad tomará (*on-policy*).

In [ ]:
from cartpole_lab.evaluation.methods import PolicySpec, build_controller
from cartpole_lab.rl.discretization import BOXES_1983
from cartpole_lab.rl.tabular import GreedyTabularPolicy, TabularConfig, train_tabular

politicas = {}  # método -> política greedy (en vivo o cargada), para las secciones siguientes
for metodo, nombre in (("q_learning", "Q-learning"), ("sarsa", "SARSA")):
    semilla = tl.exported_seed(nombre)
    registro = [r for r in tab["runs"] if r["method"] == metodo and r["discretization"] == "boxes_1983"]
    rec = next(r for r in registro if r["seed"] == semilla)
    if REENTRENAR:
        cfg = TabularConfig(method=metodo, n_episodes=tab["n_episodes"], eval_every=tab["configs"][metodo]["eval_every"])
        t0 = time.perf_counter()
        run = train_tabular(cfg, BOXES_1983, seed=semilla)
        curva = run.eval_returns.mean(axis=1)
        print(f"{nombre}, semilla {semilla}: {cfg.n_episodes} episodios en {time.perf_counter() - t0:.0f} s; "
              f"mejor política en validación: episodio {run.best_eval_episode} ({run.best_eval_mean:.0f}). "
              f"Curva: {tl.compare_with_record(curva, rec['eval_mean'])}")
        politicas[nombre] = GreedyTabularPolicy(run.best_q_table, BOXES_1983)
        tl.plot_live_vs_recorded(rec["eval_episodes"], [r["eval_mean"] for r in registro], run.eval_episodes, curva,
                                 color=METHOD_COLORS[nombre], title=f"{nombre}: 162 cajas, semilla {semilla}",
                                 xlabel="episodios de entrenamiento")
    else:
        politicas[nombre] = build_controller(PolicySpec(nombre, semilla), params)
plt.show()

La curva **no converge**: la política greedy sube y baja. Con cajas, una misma caja mezcla estados
distintos y la política "tirita". Por eso se guarda la **mejor política vista en validación** (parada
temprana), nunca elegida con el test. ¿Qué ha aprendido, y cómo controla?

In [ ]:
for nombre in ("Q-learning", "SARSA"):
    s = sanity_statistics(politicas[nombre], params)
    print(f"{nombre:<11} test (50 CI): sobrevive {s['survival_rate']:.0%}, estabilizado {s['stabilized_rate']:.0%}, "
          f"esfuerzo {s['control_effort_abs_Ns']['mean']:.0f} N·s (PID: {sanity['control_effort_abs_Ns']['mean']:.1f}), "
          f"en el límite ±10 N el {s['fraction_of_steps_at_force_limit']['mean']:.0%} del tiempo")
tl.plot_force_maps({"Q-learning": politicas["Q-learning"], "SARSA": politicas["SARSA"], "Fuzzy (escrito)": fuzzy},
                   params, title="Reglas aprendidas frente a reglas escritas (carro centrado y quieto)")
plt.show()

**Mensaje** (docs/07_08 §5): el RL tabular aprende sin modelo y redescubre reglas parecidas a las de un
experto, pero tarda miles de episodios, no converge sin parada temprana y **hace lo que pide la
recompensa** (sobrevivir), no lo que queremos: control *bang-bang* caro y carro sin centrar.

### 3.2 DQN (doc 09)

Una red neuronal $Q_\phi(s, \cdot)$ (4 → 64 → 64 → 5) recibe el estado **continuo** en lugar de una caja.
Memoria de repetición y red objetivo (Mnih et al., 2015). 100 000 pasos de entorno.

In [ ]:
from cartpole_lab.rl.dqn import DQNConfig, DQNPolicy, input_scales, q_upper_bound, train_dqn

registro_dqn = tl.load_result("rl/dqn.json")
semilla = tl.exported_seed("DQN")
runs_dqn = [r for r in registro_dqn["runs"] if r["variant"] == "dqn"]
rec = next(r for r in runs_dqn if r["seed"] == semilla)
if REENTRENAR:
    cfg = DQNConfig(**{**registro_dqn["configs"]["dqn"], "hidden": tuple(registro_dqn["configs"]["dqn"]["hidden"]),
                       "force_levels": tuple(registro_dqn["configs"]["dqn"]["force_levels"])})
    t0 = time.perf_counter()
    run = train_dqn(cfg, seed=semilla, params=params)
    curva = run.eval_returns.mean(axis=1)
    print(f"DQN, semilla {semilla}: {cfg.total_steps} pasos en {time.perf_counter() - t0:.0f} s; "
          f"mejor red en el paso {run.best_eval_step} ({run.best_eval_mean:.0f}). "
          f"Curva: {tl.compare_with_record(curva, rec['eval_mean'])}")
    politicas["DQN"] = DQNPolicy(run.best_layers, input_scales(params), cfg.force_levels)
    politicas["DQN (red final)"] = DQNPolicy(run.final_layers, input_scales(params), cfg.force_levels)
    q_reposo = run.q_at_rest
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    tl.plot_live_vs_recorded(rec["eval_steps"], [r["eval_mean"] for r in runs_dqn], run.eval_steps, curva,
                             color=METHOD_COLORS["DQN"], title=f"DQN, semilla {semilla}", xlabel="pasos de entorno",
                             ax=axes[0])
    axes[1].plot(run.eval_steps, q_reposo, color=METHOD_COLORS["DQN"], label="max Q(s = 0) en vivo")
    axes[1].axhline(q_upper_bound(cfg.gamma), color=MUTED, linestyle=":", label="máximo físico 1/(1−γ) = 100")
    axes[1].set_xlabel("pasos de entorno"); axes[1].legend(loc="lower right")
    axes[1].set_title("Sobreestimación: Q por encima de lo posible", loc="left", fontsize=11)
    plt.show()
else:
    politicas["DQN"] = build_controller(PolicySpec("DQN", semilla), params)
    politicas["DQN (red final)"] = DQNPolicy(json.loads(
        (tl.RESULTS_DIR / "policies" / "dqn" / f"seed_{semilla:02d}_final.json").read_text())["layers"],
        input_scales(params), DQNConfig().force_levels)
for nombre in ("DQN", "DQN (red final)"):
    s = sanity_statistics(politicas[nombre], params)
    print(f"{nombre:<16} test (50 CI): sobrevive {s['survival_rate']:.0%}, esfuerzo {s['control_effort_abs_Ns']['mean']:.0f} N·s")

In [ ]:
# Lo registrado con las 10 semillas de cada variante (docs/09): Double DQN corrige la sobreestimación,
# pero no el colapso. El diagnóstico apunta a la diferencia diminuta entre acciones.
for variante, nombre in (("dqn", "DQN"), ("double_dqn", "Double DQN")):
    r = registro_dqn["summary"][variante]
    print(f"{nombre:<11} validación final {r['final_greedy_validation']['mean']:5.0f} ± {r['final_greedy_validation']['std']:3.0f}"
          f" | mejor vista {r['best_checkpoint_validation']['mean']:5.0f}"
          f" | Q(s=0) final {r['q_at_rest_final']['mean']:5.1f} (cota 100)"
          f" | diferencia relativa entre acciones al final {r['relative_action_gap_last_eval']['mean']:.4f}")
display(Image(filename=str(tl.figure_path("09_diagnostico_dqn.png")), width=900))

**Mensaje** (docs/09 §5): una red aprende mucho más rápido que una tabla, pero **no converge**: sin guardar
la mejor red, la final es inservible. **Diagnosticar antes de "arreglar"**: la sobreestimación era real
y Double DQN la corrige, pero no era la causa del colapso. Con recompensa constante, la diferencia entre
acciones es diminuta frente al propio valor de Q.

### 3.3 Actor-crítico A2C + GAE con acción continua (doc 10)

Dos redes: un **actor** $\pi_\theta(F \mid s)$ que elige una fuerza **continua**
($F = 10\tanh u$, $u \sim \mathcal N(\mu(s), \sigma^2)$) y un **crítico** $V_w(s)$ que juzga sus
acciones con la ventaja GAE (Mnih et al., 2016; Schulman et al., 2016). 8 entornos en paralelo,
307 200 pasos.

In [ ]:
from cartpole_lab.rl.actor_critic import ACConfig, ACPolicy, train_a2c

registro_ac = tl.load_result("rl/actor_critic.json")
semilla = tl.exported_seed("Actor-crítico")
rec = next(r for r in registro_ac["runs"] if r["seed"] == semilla)
if REENTRENAR:
    cfg = ACConfig(**{**registro_ac["config"], "hidden": tuple(registro_ac["config"]["hidden"])})
    t0 = time.perf_counter()
    run = train_a2c(cfg, seed=semilla, params=params)
    curva = run.eval_returns.mean(axis=1)
    print(f"Actor-crítico, semilla {semilla}: {cfg.total_steps} pasos en {time.perf_counter() - t0:.0f} s; "
          f"primera red con {run.best_eval_mean:.0f} en el paso {run.best_eval_step}. "
          f"Curva: {tl.compare_with_record(curva, rec['eval_mean'])}")
    politicas["Actor-crítico"] = ACPolicy(run.best_layers, input_scales(params), params.force_mag)
    politicas["Actor-crítico (red final)"] = ACPolicy(run.final_layers, input_scales(params), params.force_mag)
    tl.plot_live_vs_recorded(rec["eval_steps"], [r["eval_mean"] for r in registro_ac["runs"]], run.eval_steps, curva,
                             color=METHOD_COLORS["Actor-crítico"], title=f"Actor-crítico, semilla {semilla}",
                             xlabel="pasos de entorno")
else:
    politicas["Actor-crítico"] = build_controller(PolicySpec("Actor-crítico", semilla), params)
    politicas["Actor-crítico (red final)"] = build_controller(PolicySpec("Actor-crítico (red final)", semilla), params)
for nombre in ("Actor-crítico", "Actor-crítico (red final)"):
    s = sanity_statistics(politicas[nombre], params)
    print(f"{nombre:<26} test (50 CI): sobrevive {s['survival_rate']:.0%}, estabilizado {s['stabilized_rate']:.0%}, "
          f"esfuerzo {s['control_effort_abs_Ns']['mean']:.2f} N·s, |x| final {s['last_second_max_abs_x_m']['mean']:.2f} m")
plt.show()

In [ ]:
tl.plot_force_maps({"Actor-crítico": politicas["Actor-crítico"], "Fuzzy": fuzzy, "LQR": lqr}, params,
                   title="El actor-crítico redescubre un PD saturado (carro centrado y quieto)")
tray = tl.simulate({"PID": pid, "LQR": lqr, "Actor-crítico": politicas["Actor-crítico"],
                    "Actor-crítico (red final)": politicas["Actor-crítico (red final)"]},
                   initial_state=REFERENCE_HARD_INITIAL_STATE)
tl.plot_trajectories(tray, params, title="Fuera de la distribución de entrenamiento: estado difícil de referencia")
plt.show()

**Mensaje** (docs/10 §6): el actor-crítico con acción continua es el RL que mejor funciona (en el
registro converge en 10/10 semillas, con esfuerzo de controlador clásico) y redescubre un PD saturado.
Pero **solo sabe lo que ha vivido**: fuera de la distribución de entrenamiento (carro desplazado, poste
muy inclinado) falla donde el LQR o el MPC no fallan. Y **hace lo que pide la recompensa**: poste vertical
sí, carro centrado no.

## 4. Comparación (docs/12 y 13)

Protocolo **declarado antes de ver resultados** (docs/12), con sus dos enmiendas registradas: escenario
nominal, impulso lateral en t = 5 s, regiones de atracción en dos mallas 21 × 21 y coste de aprendizaje.
Los clásicos son un solo controlador; cada método de RL se resume con la media ± desviación típica
**entre 10 semillas**.

Con `REEVALUAR = True`, la celda siguiente relanza la evaluación completa y las figuras, que sobrescriben
`results/metrics` y `results/figures`. Con `False`, se cargan los resultados registrados.

In [ ]:
import subprocess
import sys

from cartpole_lab.paths import REPO_ROOT

if REEVALUAR:
    for script in ("scripts/evaluate_all.py", "scripts/make_figures.py"):
        t0 = time.perf_counter()
        subprocess.run([sys.executable, script], cwd=REPO_ROOT, check=True)
        print(f"{script}: {time.perf_counter() - t0:.0f} s")
else:
    print("Se usan los resultados registrados en results/metrics/comparison.json")

In [ ]:
# Resultados de la evaluación pre-registrada (docs/12): 8 métodos, 50 CI nominales, 8 impulsos,
# 2 mallas de 21 × 21 estados iniciales; RL con 10 semillas por método.
comparacion = tl.load_result("metrics/comparison.json")
print(f"{comparacion['n_jobs']} tareas · protocolo: {comparacion['protocol']} · "
      f"impulsos {comparacion['impulses_Ns']} N·s · malla {comparacion['grid_n']} × {comparacion['grid_n']}")
display(Markdown(tl.summary_table_markdown()))

In [ ]:
# Figuras finales (scripts/make_figures.py), generadas a partir de comparison.json.
for nombre in ("13_resumen_nominal.png", "13_robustez_impulso.png", "13_regiones_atraccion.png",
               "13_coste_aprendizaje.png"):
    display(Image(filename=str(tl.figure_path(nombre)), width=1000))

In [ ]:
# Demostración en vivo del escenario de robustez (docs/12 §2): misma CI (semilla 1000, distribución
# oficial) y un impulso lateral de 1.0 N·s en t = 5 s para los 8 métodos. RL: la semilla exportada.
from cartpole_lab.evaluation.metrics import episode_metrics
from cartpole_lab.evaluation.protocol import IMPULSE_STEP

J = 1.0  # N·s: por debajo del J₅₀ de los cuatro clásicos en el registro (1.25–1.5 N·s, doc 13)
tarea_impulso = CartPoleTask(action_mode="continuous", disturbance=ForcePulse(IMPULSE_STEP, 1, J / params.tau))
controladores = tl.method_controllers(params)
t0 = time.perf_counter()
try:
    tray_impulso = tl.simulate(controladores, seed=1000, task=tarea_impulso)
finally:
    tarea_impulso.close()
print(f"8 episodios en {time.perf_counter() - t0:.1f} s\n")
print(f"{'método':<15}{'sobrevive':>10}{'recuperación del poste [s]':>30}{'esfuerzo [N·s]':>17}")
for nombre, tr in tray_impulso.items():
    m = episode_metrics(tr, impulse_step=IMPULSE_STEP)
    rec = "—" if m["recovery_time_s"] is None else f"{m['recovery_time_s']:.2f}"
    print(f"{nombre:<15}{str(m['survived']):>10}{rec:>30}{m['effort_abs_Ns']:>17.2f}")

t_imp = IMPULSE_STEP * params.tau
clasicos = ("PID", "LQR", "MPC", "Fuzzy")
tl.plot_trajectories({k: v for k, v in tray_impulso.items() if k in clasicos}, params, impulse_time=t_imp,
                     title=f"Control clásico: impulso de {J} N·s en t = {t_imp:.0f} s")
tl.plot_trajectories({k: v for k, v in tray_impulso.items() if k not in clasicos}, params, impulse_time=t_imp,
                     title="Aprendizaje por refuerzo (semilla exportada): mismo impulso")
plt.show()

### Mensajes para la charla (docs/13 §8, sobre la evaluación registrada)

1. **Si hay modelo, úsalo.** LQR y MPC controlan con precisión, gastando muy poco y sin un solo
   episodio de prueba. El MPC además amplía la región de recuperación, porque planifica con las
   restricciones.
2. **El RL aprende a no caerse, no a controlar.** Q-learning, SARSA y DQN mantienen el poste arriba
   oscilando, con 200 veces más esfuerzo. El actor-crítico se acerca a un clásico en el poste, pero deja
   el carro donde caiga, porque la recompensa no lo pide.
3. **El RL solo sabe lo que ha vivido.** Fuera de la distribución de entrenamiento, sus regiones de
   recuperación son de 1.5 a 10 veces menores que las de los clásicos.
4. **Cuidado con cómo se mide la robustez.** Que una política sobreviva a un impulso mayor puede ser
   suerte de fase, no robustez (doc 13 §3).
5. **Cuidado con cómo se mide la eficiencia.** Un PID "clásico" sintonizado por fuerza bruta usó más
   simulaciones que cualquier RL (doc 13 §5).

Limitaciones de la comparación (horizonte de 10 s, un solo tipo de perturbación, cortes 2D de la región
de atracción, 10 semillas…): docs/13 §9.